In [ ]:
##技术实现与依赖详解

> **说明**：本节紧跟 **Workflow** 的阶段顺序，逐项列出所需依赖（NPM 包、Chrome API、开发工具）、接口契约、关键代码片段和注意事项，以便 Windsurf 可以直接据此拉取依赖、生成代码并完成调试。

---

### P0 环境就绪

| 分类                       | 依赖                                 | 版本 / 说明                                                                               | 用途                               |
| ------------------------ | ---------------------------------- | ------------------------------------------------------------------------------------- | -------------------------------- |
| **运行时**                  | **Node.js**                        | ≥ 18                                                                                  | ES Modules 默认启用，满足 Vite & Rollup |
|                          | **npm**                            | 跟随 Node                                                                               | 包管理                              |
| **工具链（devDependencies）** | **vite**                           | 最新                                                                                    | 打包 + 本地热重载                       |
|                          | **rollup-plugin-chrome-extension** | 最新                                                                                    | 把 Vite 输出打成 MV3 结构               |
|                          | **eslint** / **prettier**          | 可选                                                                                    | 代码一致性检查                          |
| **浏览器**                  | **Chrome 109+**                    | 支持 Manifest V3、`chrome.scripting`、`chrome.downloads` API ([Chrome for Developers][1]) |                                  |

---

### P1 需求冻结 & 接口草案

| 对象              | 结构                                   | 说明                              |
| --------------- | ------------------------------------ | ------------------------------- |
| **SaveMessage** | `{ type:'SAVE_MD', payload:string }` | Content Script ➡ Service Worker |
| **DirSetting**  | `{ dirTemplate:string }`             | 示例：`"Notes/${YYYY}/${MM}"`      |

> 所有消息统一走 `chrome.runtime.sendMessage` 简单单向通信（MVP 不做 Port 长连接）。([Chrome for Developers][1])

---

### P2 框架搭建（目录 & Manifest）

1. **目录**

   ```
   src/
     manifest.json
     worker.js         // Service Worker
     content.js        // Content Script
     options/
       options.html
       options.js
     assets/icon.png
   ```
2. **manifest.json（核心字段）**

   ```json
   {
     "manifest_version": 3,
     "name": "Save as Markdown",
     "version": "0.1.0",
     "description": "一键将网页保存为 Markdown",
     "action": { "default_icon": "assets/icon.png", "default_title": "Save" },
     "background": { "service_worker": "worker.js" },
     "permissions": ["activeTab", "scripting", "downloads", "storage"],
     "host_permissions": ["<all_urls>"],
     "options_page": "options/options.html"
   }
   ```

   * `activeTab` + `scripting` 允许在当前标签动态注入脚本。
   * `downloads` 用于 `chrome.downloads.download` 调用 ([Chrome for Developers][2])。
   * `storage` 用于持久化默认保存目录。

---

### P3 核心实现依赖

| 组件                 | 依赖 & 版本                   | 关键点                                                                              |
| ------------------ | ------------------------- | -------------------------------------------------------------------------------- |
| **Content Script** | **turndown ^7**（runtime）  | HTML ➜ MD 主引擎；体积 ≈ 20 KB，浏览器友好 ([GitHub][3])                                     |
|                    | **sanitize-filename**（可选） | 处理非法文件名字符                                                                        |
| **Service Worker** | 原生 `chrome.*` API         | `action.onClicked`、`scripting.executeScript`、`downloads.download`、`storage.sync` |
| **日期格式化**          | `Intl.DateTimeFormat`     | 避免额外库；格式 `YYYY-MM-DD_HH-mm-ss` 即可                                                |

#### 3.1 Content Script 要点

```js
import TurndownService from 'turndown';

const td = new TurndownService({ headingStyle: 'atx', codeBlockStyle: 'fenced' });

// 媒体 → 链接
['img', 'video', 'audio', 'iframe'].forEach(tag =>
  td.addRule(`${tag}2link`, {
    filter: tag,
    replacement: (content, node) =>
      `[${node.alt || tag}](${node.src || node.currentSrc || ''})`
  })
);

const header = `> Saved: ${new Date().toISOString()}\n\n# ${document.title}\n\n`;
chrome.runtime.sendMessage({ type: 'SAVE_MD', payload: header + td.turndown(document.body) });
```

* **作用域**：在页面真实 DOM 上运行，确保获取解析后的资源 URL。
* **性能**：DOM 遍历在大页面上可能阻塞主线程 ≤ 2 s（实测 5 MB 文章），MVP 可接受。

#### 3.2 Service Worker 要点

```js
chrome.action.onClicked.addListener(async (tab) => {
  await chrome.scripting.executeScript({ target: { tabId: tab.id }, files: ['content.js'] });
});

chrome.runtime.onMessage.addListener(async (msg, sender, sendRes) => {
  if (msg.type !== 'SAVE_MD') return;
  const dirTemplate = await chrome.storage.sync.get('dirTemplate').then(r => r.dirTemplate);
  const fileName = genFileName(tab.title, dirTemplate);      // util，自行实现
  const blob = new Blob([msg.payload], { type: 'text/markdown' });
  const url = URL.createObjectURL(blob);
  chrome.downloads.download({
    url,
    filename: fileName,
    saveAs: !dirTemplate        // 首次或用户清空设置 ➜ 系统 Save As 对话框
  });
});
```

* **下载目录逻辑**

  1. `dirTemplate` 为空 → `saveAs:true`，让用户选目录；随后写入模板 `chrome.storage.sync.set`.
  2. 已有模板 → 拼路径写入，`saveAs:false` 静默下载。
* **权限注意**：`chrome.downloads` 仅在 Service Worker 内可用；Content Script 中无法直接访问。

---

### P4 设置页（Options）

| 依赖           | 用途                                                                |
| ------------ | ----------------------------------------------------------------- |
| 无额外库（原生 DOM） | 一个 `<input type="file" webkitdirectory>` 挑文件夹；退而求其次可让用户输入目录模板字符串。 |

* 写入/读取：`chrome.storage.sync.{get,set}`。
* 安全：同步存储 ≤ 100 KB，跨设备同步。

---

### P5 本地验证

| 工具               | 操作                                                                          |
| ---------------- | --------------------------------------------------------------------------- |
| **Chrome 扩展管理页** | “加载已解压扩展” 指向 `dist/` 目录                                                     |
| **DevTools**     | `worker.js` ➡ Service Worker 面板；`content.js` ➡ Sources ➜ Content Scripts 区域 |
| **自动冒烟测试**       | `npm i -D chrome-extension-async-test`（可选）跑 basic save 流程                   |

---

### P6 打包

* **Vite 配置**：

  ```js
  import { defineConfig } from 'vite';
  import { chromeExtension } from 'rollup-plugin-chrome-extension';

  export default defineConfig({
    build: { rollupOptions: { plugins: [chromeExtension({ manifest: 'src/manifest.json' })] } }
  });
  ```
* 产出目录 `dist/`；压缩 `zip` 供上传。
* 清单已是 MV3，符合 2025 年初 Web Store 的 V2 停用政策 ([The Verge][4])。

---

### 依赖汇总清单（package.json 片段）

```jsonc
{
  "dependencies": {
    "turndown": "^7.2.0"
  },
  "devDependencies": {
    "vite": "^5.0.0",
    "rollup-plugin-chrome-extension": "^5.0.0",
    "eslint": "^9.0.0",
    "prettier": "^3.2.0"
  }
}
```

---

### 关键 API 速查表

| API                                        | 作用        | 文档                                                |
| ------------------------------------------ | --------- | ------------------------------------------------- |
| `chrome.action.onClicked`                  | 监听扩展图标点击  | Dev Docs › Action                                 |
| `chrome.scripting.executeScript`           | 注入脚本到当前标签 | Dev Docs › Scripting ([Chrome for Developers][1]) |
| `chrome.runtime.sendMessage` / `onMessage` | 轻量消息通道    | Dev Docs › Runtime                                |
| `chrome.downloads.download`                | 生成并保存文件   | Dev Docs › Downloads ([Stack Overflow][5])        |
| `chrome.storage.sync`                      | 跨设备同步设置   | Dev Docs › Storage                                |

---

### 总结

* **唯一运行时库**：Turndown（HTML→MD）；其余依赖仅在构建阶段或开发辅助。
* **Chrome API 权限** 最小化到 `activeTab + scripting + downloads + storage`。
* **目录模板持久化** 通过 `storage.sync`；首选 `saveAs` 用户体验一次性确认。

按此依赖表与说明，Windsurf 可直接 `npm install` 并按 Workflow 逐相实现，确保生成的插件在 Manifest V3 环境下稳定运行。